In [ ]:
import os
import numpy as np
import pandas as pd

PROCESSED_PATH = "processed_transactions.parquet"
RAW_PATH = "/Users/yuliia/Documents/Fraud-Detection/final_data.csv"

def build_processed_df(raw_path: str) -> pd.DataFrame:
    df = pd.read_csv(raw_path)

    date_cols = ["trn_date", "person_creation_date", "person_first_transaction"]
    for col in date_cols:
        df[col] = pd.to_datetime(df[col])

    df["date"] = df["trn_date"].dt.date
    df["week"] = df["trn_date"].dt.to_period("W").dt.start_time
    df["hour"] = df["trn_date"].dt.hour
    df["dow"] = df["trn_date"].dt.day_name()

    df["trn_id"] = df["trn_id"].astype("string")
    df["place_id"] = df["place_id"].astype("string")
    df["partner_id"] = df["partner_id"].astype("string")
    df["trn_reg_user_code"] = df["trn_reg_user_code"].astype("string")

    df["waiter_id"] = (
        df["place_id"].astype("string").fillna("")
        + "_"
        + df["trn_reg_user_code"].astype("string").fillna("")
    )

    df = df[df["person_first_transaction"] >= pd.Timestamp("2010-01-01", tz="UTC")]

    # транзакції, де використовувались бофони, але вони не врахувались в discount_amount
    error_trn = df[(df["discount_amount"] == 0) & (df["bonusses_used"] != 0)]["trn_id"].values
    df = df[~df["trn_id"].isin(error_trn)]

    # забрати транзакції з відємним чеком
    df = df[df["total_amount"] >= 0]

    df["gross_amount"] = df["total_amount"] + df["discount_amount"]

    # зробити bonusses_used додатним
    df["bonusses_used"] = (-1) * df["bonusses_used"]

    # ceil discount_amount
    df["discount_amount"] = np.ceil(df["discount_amount"])

    # прибираю проблемні транзакції, де сума бонусів не врахувалась в суму дисконту
    df = df[df["discount_amount"] >= df["bonusses_used"]]

    df["promo_used"] = df["discount_amount"] - df["bonusses_used"]

    df["is_first_trn"] = df["trn_count_before"] == 0
    df["bonus_used_flag"] = df["bonusses_used"] > 0

    df = df.sort_values(["person_id", "trn_date"])

    df["time_since_prev_trn_hours"] = (
        df.groupby("person_id")["trn_date"]
          .diff()
          .dt.total_seconds() / 3600
    )

    df = df[df["person_first_transaction"] >= pd.Timestamp("2024-01-01", tz="UTC")]

    return df

if os.path.exists(PROCESSED_PATH):
    df = pd.read_parquet(PROCESSED_PATH)
else:
    df = build_processed_df(RAW_PATH)
    df.to_parquet(PROCESSED_PATH, index=False)